In [2]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("PredictiveMaintenance")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

dataset = Path("/home/jovyan/work/data/raw/sensor_data.csv")

print("Spark version:", spark.version)
print("Dataset found:", dataset.is_file())
print("Test count:", spark.range(1000).count())

Spark version: 4.2.0
Dataset found: True
Test count: 1000


In [3]:
import csv

with dataset.open(newline="") as file:
    reader = csv.reader(file)
    print("Columns:", next(reader))
    for _, row in zip(range(5), reader):
        print(row)

print(f"\nFile size: {dataset.stat().st_size / 1024**3:.2f} GiB")

Columns: ['when', 'hwid', 'metric', 'value']
['1601510485159', 'SW-065', 'SA4', '0']
['1601510485159', 'SW-065', 'SA3', '0']
['1601510485159', 'SW-065', 'SA2', '0']
['1601510485159', 'SW-065', 'S34', '0']
['1601510485159', 'SW-065', 'S33', '1']

File size: 4.76 GiB


In [4]:
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DoubleType
)

# Explicit types avoid scanning the CSV to infer its schema.
schema = StructType([
    StructField("when", LongType(), nullable=True),
    StructField("hwid", StringType(), nullable=True),
    StructField("metric", StringType(), nullable=True),
    StructField("value", DoubleType(), nullable=True),
])

# FAILFAST raises an error if Spark encounters a malformed record.
# Reading is lazy: this defines the DataFrame without loading it all.
raw_data = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .schema(schema)
    .csv(str(dataset))
)

raw_data.printSchema()
raw_data.show(5, truncate=False)

root
 |-- when: long (nullable = true)
 |-- hwid: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)

+-------------+------+------+-----+
|when         |hwid  |metric|value|
+-------------+------+------+-----+
|1601510485159|SW-065|SA4   |0.0  |
|1601510485159|SW-065|SA3   |0.0  |
|1601510485159|SW-065|SA2   |0.0  |
|1601510485159|SW-065|S34   |0.0  |
|1601510485159|SW-065|S33   |1.0  |
+-------------+------+------+-----+
only showing top 5 rows


In [5]:
from pyspark.sql import functions as F
from time import perf_counter

start = perf_counter()

# Calculate several checks together in one aggregation.
summary = raw_data.agg(
    F.count("*").alias("total_rows"),
    F.min("when").alias("first_timestamp_ms"),
    F.max("when").alias("last_timestamp_ms"),
    *[
        F.count(F.when(F.col(name).isNull(), 1))
        .alias(f"missing_{name}")
        for name in raw_data.columns
    ],
    F.count(F.when(F.isnan("value"), 1)).alias("nan_value"),
)

# Convert milliseconds to readable timestamps, using our UTC setting.
summary = summary.select(
    "*",
    F.timestamp_millis("first_timestamp_ms").alias("first_time_utc"),
    F.timestamp_millis("last_timestamp_ms").alias("last_time_utc"),
)

summary.show(truncate=False)

print(f"Scan completed in {perf_counter() - start:.1f} seconds")

+----------+------------------+-----------------+------------+------------+--------------+-------------+---------+-----------------------+-----------------------+
|total_rows|first_timestamp_ms|last_timestamp_ms|missing_when|missing_hwid|missing_metric|missing_value|nan_value|first_time_utc         |last_time_utc          |
+----------+------------------+-----------------+------------+------------+--------------+-------------+---------+-----------------------+-----------------------+
|142102770 |1601510422405     |1617235197606    |0           |0           |0             |0            |0        |2020-10-01 00:00:22.405|2021-03-31 23:59:57.606|
+----------+------------------+-----------------+------------+------------+--------------+-------------+---------+-----------------------+-----------------------+

Scan completed in 65.2 seconds


In [6]:
device_summary = (
    raw_data
    .groupBy("hwid")
    .agg(
        F.count("*").alias("rows"),
        F.countDistinct("metric").alias("unique_metrics"),
        F.timestamp_millis(F.min("when")).alias("first_time_utc"),
        F.timestamp_millis(F.max("when")).alias("last_time_utc"),
    )
    .orderBy("hwid")
)

device_summary.show(truncate=False)

+------+--------+--------------+-----------------------+-----------------------+
|hwid  |rows    |unique_metrics|first_time_utc         |last_time_utc          |
+------+--------+--------------+-----------------------+-----------------------+
|SW-065|27576413|132           |2020-10-01 00:01:25.159|2021-01-13 12:14:16.418|
|SW-088|42095692|133           |2020-10-01 00:00:58.92 |2021-03-31 23:59:57.606|
|SW-106|26187749|133           |2020-10-01 00:00:22.405|2021-03-27 07:28:36.654|
|SW-115|46242916|133           |2020-10-01 00:02:12.889|2021-03-31 23:59:48.764|
+------+--------+--------------+-----------------------+-----------------------+



In [7]:
alarms = raw_data.filter(F.col("metric").isin("A5", "A9"))

invalid_alarm = (
    F.col("value").isNull()
    | F.isnan("value")
    | (F.col("value") < 0)
    | (F.col("value") > 65535)
    | (F.col("value") != F.floor("value"))
)

alarm_summary = (
    alarms.groupBy("hwid", "metric")
    .agg(
        F.count("*").alias("records"),
        F.min("value").alias("min_code"),
        F.max("value").alias("max_code"),
        F.countDistinct("value").alias("distinct_codes"),
        F.sum(
            F.when(invalid_alarm, 1).otherwise(0)
        ).alias("invalid_codes"),
        F.timestamp_millis(F.min("when")).alias("first_time_utc"),
        F.timestamp_millis(F.max("when")).alias("last_time_utc"),
    )
    .orderBy("hwid", "metric")
)

alarm_summary.show(truncate=False)

+------+------+-------+--------+--------+--------------+-------------+-----------------------+-----------------------+
|hwid  |metric|records|min_code|max_code|distinct_codes|invalid_codes|first_time_utc         |last_time_utc          |
+------+------+-------+--------+--------+--------------+-------------+-----------------------+-----------------------+
|SW-065|A5    |155    |0.0     |0.0     |1             |0            |2020-10-01 00:16:51.319|2021-01-13 00:21:05.082|
|SW-065|A9    |155    |0.0     |0.0     |1             |0            |2020-10-01 00:16:51.319|2021-01-13 00:21:05.082|
|SW-088|A5    |217    |0.0     |26688.0 |5             |0            |2020-10-01 00:06:22.759|2021-03-31 16:09:21.608|
|SW-088|A9    |305    |0.0     |16384.0 |13            |0            |2020-10-01 00:06:22.759|2021-03-31 15:13:51.329|
|SW-106|A5    |92     |0.0     |0.0     |1             |0            |2020-10-01 00:14:56.766|2021-03-27 00:53:19.156|
|SW-106|A9    |92     |0.0     |0.0     |1      

In [8]:
decoded_alarms = alarms.withColumn(
    "alarm_code", F.col("value").cast("long")
)

for position in (6, 7, 8):
    mask = 1 << (position - 1)
    decoded_alarms = decoded_alarms.withColumn(
        f"bit_{position}",
        F.col("alarm_code").bitwiseAND(mask) != 0
    )

decoded_alarms = decoded_alarms.withColumn(
    "overheating",
    F.col("bit_6") | F.col("bit_7") | F.col("bit_8")
)

# Small code-frequency table; no missing records are filled.
code_summary = (
    decoded_alarms
    .groupBy(
        "hwid", "metric", "alarm_code",
        "bit_6", "bit_7", "bit_8", "overheating"
    )
    .count()
    .orderBy("hwid", "metric", "alarm_code")
)

code_summary.show(100, truncate=False)

+------+------+----------+-----+-----+-----+-----------+-----+
|hwid  |metric|alarm_code|bit_6|bit_7|bit_8|overheating|count|
+------+------+----------+-----+-----+-----+-----------+-----+
|SW-065|A5    |0         |false|false|false|false      |155  |
|SW-065|A9    |0         |false|false|false|false      |155  |
|SW-088|A5    |0         |false|false|false|false      |182  |
|SW-088|A5    |64        |false|true |false|true       |17   |
|SW-088|A5    |192       |false|true |true |true       |4    |
|SW-088|A5    |18496     |false|true |false|true       |11   |
|SW-088|A5    |26688     |false|true |false|true       |3    |
|SW-088|A9    |0         |false|false|false|false      |140  |
|SW-088|A9    |2         |false|false|false|false      |3    |
|SW-088|A9    |4         |false|false|false|false      |1    |
|SW-088|A9    |64        |false|true |false|true       |102  |
|SW-088|A9    |128       |false|false|true |true       |8    |
|SW-088|A9    |192       |false|true |true |true       

In [9]:
from pyspark.sql.window import Window

alarm_window = Window.partitionBy("hwid", "metric").orderBy("when")

alarm_timing = (
    decoded_alarms
    .withColumn("previous_time", F.lag("when").over(alarm_window))
    .withColumn("previous_code", F.lag("alarm_code").over(alarm_window))
    .withColumn(
        "gap_hours",
        (F.col("when") - F.col("previous_time")) / 3_600_000
    )
)

timing_summary = (
    alarm_timing.groupBy("hwid", "metric")
    .agg(
        F.count("*").alias("records"),
        F.min("gap_hours").alias("min_gap_hours"),
        F.expr(
            "percentile_approx(gap_hours, 0.5)"
        ).alias("median_gap_hours"),
        F.max("gap_hours").alias("max_gap_hours"),
        F.sum(
            F.when(
                F.col("alarm_code") == F.col("previous_code"), 1
            ).otherwise(0)
        ).alias("repeated_code_records"),
    )
    .orderBy("hwid", "metric")
)

timing_summary.show(truncate=False)

+------+------+-------+---------------------+------------------+------------------+---------------------+
|hwid  |metric|records|min_gap_hours        |median_gap_hours  |max_gap_hours     |repeated_code_records|
+------+------+-------+---------------------+------------------+------------------+---------------------+
|SW-065|A5    |155    |0.06387833333333333  |18.274605277777777|71.98137416666667 |154                  |
|SW-065|A9    |155    |0.06387833333333333  |18.274605277777777|71.98137416666667 |154                  |
|SW-088|A5    |217    |0.0053952777777777775|23.6534           |224.98014         |186                  |
|SW-088|A9    |305    |0.0056727777777777775|11.572233888888888|225.46082611111112|244                  |
|SW-106|A5    |92     |0.1309202777777778   |23.989214444444446|1405.1200377777777|91                   |
|SW-106|A9    |92     |0.1309202777777778   |23.989214444444446|1405.1200377777777|91                   |
|SW-115|A5    |187    |0.008365             |2

In [10]:
sw088_timeline = (
    alarm_timing
    .filter(F.col("hwid") == "SW-088")
    .withColumn(
        "previous_overheating",
        F.lag("overheating").over(alarm_window)
    )
)

# Show the first reading and changes in the observed overheating flag.
# These are observed transitions, not verified incident start/end times.
observed_changes = (
    sw088_timeline
    .filter(
        F.col("previous_time").isNull()
        | (F.col("overheating") != F.col("previous_overheating"))
    )
    .select(
        "metric",
        F.timestamp_millis("when").alias("time_utc"),
        "alarm_code",
        "previous_overheating",
        "overheating",
        F.round("gap_hours", 2).alias("gap_hours"),
    )
    .orderBy("metric", "time_utc")
)

observed_changes.show(100, truncate=False)

+------+-----------------------+----------+--------------------+-----------+---------+
|metric|time_utc               |alarm_code|previous_overheating|overheating|gap_hours|
+------+-----------------------+----------+--------------------+-----------+---------+
|A5    |2020-10-01 00:06:22.759|18496     |NULL                |true       |NULL     |
|A5    |2020-10-06 07:57:08.242|0         |true                |false      |0.77     |
|A5    |2020-12-16 09:44:32.224|64        |false               |true       |9.63     |
|A5    |2020-12-16 10:01:45.046|0         |true                |false      |0.29     |
|A5    |2021-01-26 09:44:27.942|64        |false               |true       |9.44     |
|A5    |2021-01-26 11:19:42.042|0         |true                |false      |0.62     |
|A5    |2021-02-02 15:19:05.465|64        |false               |true       |15.1     |
|A5    |2021-02-02 17:17:55.68 |0         |true                |false      |1.98     |
|A5    |2021-02-03 16:11:45.931|64        |

In [11]:
# Daily recording coverage for SW-088.
# No records on a day will remain absent—not be labelled "normal".
daily_coverage = (
    raw_data
    .filter(F.col("hwid") == "SW-088")
    .withColumn("date", F.to_date(F.timestamp_millis("when")))
    .groupBy("date")
    .agg(
        F.count("*").alias("sensor_records"),
        F.countDistinct("metric").alias("metrics_recorded"),
        F.sum(
            F.when(F.col("metric") == "A5", 1).otherwise(0)
        ).alias("a5_records"),
        F.sum(
            F.when(F.col("metric") == "A9", 1).otherwise(0)
        ).alias("a9_records"),
    )
)

daily_coverage.filter(
    F.col("date").between("2021-03-15", "2021-03-31")
).orderBy("date").show(20, truncate=False)

+----------+--------------+----------------+----------+----------+
|date      |sensor_records|metrics_recorded|a5_records|a9_records|
+----------+--------------+----------------+----------+----------+
|2021-03-15|281248        |132             |1         |2         |
|2021-03-16|268734        |132             |1         |2         |
|2021-03-17|288499        |132             |1         |2         |
|2021-03-18|260247        |131             |0         |1         |
|2021-03-19|287015        |132             |1         |2         |
|2021-03-20|264939        |132             |1         |2         |
|2021-03-21|8218          |130             |0         |0         |
|2021-03-29|175004        |133             |6         |6         |
|2021-03-30|251354        |132             |3         |6         |
|2021-03-31|282831        |132             |5         |5         |
+----------+--------------+----------------+----------+----------+



In [12]:
calendar = spark.sql("""
    SELECT explode(
        sequence(DATE '2020-10-01', DATE '2021-03-31', INTERVAL 1 DAY)
    ) AS date
""")

coverage_calendar = (
    calendar
    .join(daily_coverage, on="date", how="left")
    .withColumn(
        "no_device_records",
        F.col("sensor_records").isNull()
    )
)

coverage_calendar.filter(
    F.col("no_device_records")
).orderBy("date").show(200, truncate=False)

+----------+--------------+----------------+----------+----------+-----------------+
|date      |sensor_records|metrics_recorded|a5_records|a9_records|no_device_records|
+----------+--------------+----------------+----------+----------+-----------------+
|2021-03-22|NULL          |NULL            |NULL      |NULL      |true             |
|2021-03-23|NULL          |NULL            |NULL      |NULL      |true             |
|2021-03-24|NULL          |NULL            |NULL      |NULL      |true             |
|2021-03-25|NULL          |NULL            |NULL      |NULL      |true             |
|2021-03-26|NULL          |NULL            |NULL      |NULL      |true             |
|2021-03-27|NULL          |NULL            |NULL      |NULL      |true             |
|2021-03-28|NULL          |NULL            |NULL      |NULL      |true             |
+----------+--------------+----------------+----------+----------+-----------------+



In [13]:
hourly_coverage = (
    raw_data
    .filter(
        (F.col("hwid") == "SW-088")
        & (F.col("when") >= 1616198400000)  # March 20, UTC
        & (F.col("when") < 1617062400000)   # March 30, UTC
    )
    .withColumn("timestamp", F.timestamp_millis("when"))
    .groupBy(F.date_trunc("hour", "timestamp").alias("hour_utc"))
    .agg(
        F.count("*").alias("records"),
        F.min("timestamp").alias("first_reading"),
        F.max("timestamp").alias("last_reading"),
    )
    .orderBy("hour_utc")
)

hourly_coverage.show(100, truncate=False)

+-------------------+-------+-----------------------+-----------------------+
|hour_utc           |records|first_reading          |last_reading           |
+-------------------+-------+-----------------------+-----------------------+
|2021-03-20 00:00:00|14771  |2021-03-20 00:00:10.682|2021-03-20 00:59:58.703|
|2021-03-20 01:00:00|5589   |2021-03-20 01:00:26.599|2021-03-20 01:31:52.284|
|2021-03-20 02:00:00|13886  |2021-03-20 02:04:58.226|2021-03-20 02:59:49.123|
|2021-03-20 03:00:00|14207  |2021-03-20 03:00:16.163|2021-03-20 03:59:38.509|
|2021-03-20 04:00:00|6871   |2021-03-20 04:00:11.05 |2021-03-20 04:28:07.567|
|2021-03-20 05:00:00|5728   |2021-03-20 05:02:57.186|2021-03-20 05:59:57.986|
|2021-03-20 06:00:00|12671  |2021-03-20 06:00:00.885|2021-03-20 06:59:55.722|
|2021-03-20 07:00:00|13493  |2021-03-20 07:00:24.807|2021-03-20 07:59:59.343|
|2021-03-20 08:00:00|14076  |2021-03-20 08:00:26.643|2021-03-20 08:59:42.706|
|2021-03-20 09:00:00|6216   |2021-03-20 09:00:18.812|2021-03-20 

In [14]:
sw088_path = "/home/jovyan/work/data/processed/sw088_raw.parquet"

(
    raw_data
    .filter(F.col("hwid") == "SW-088")
    .write
    .mode("errorifexists")
    .parquet(sw088_path)
)

sw088_data = spark.read.parquet(sw088_path)

sw088_data.printSchema()

root
 |-- when: long (nullable = true)
 |-- hwid: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: double (nullable = true)



In [15]:
metric_summary = (
    sw088_data
    .groupBy("metric")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct(
            F.to_date(F.timestamp_millis("when"))
        ).alias("days_with_readings"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value"),
        F.timestamp_millis(F.min("when")).alias("first_time_utc"),
        F.timestamp_millis(F.max("when")).alias("last_time_utc"),
    )
)

# Initial candidates from your original analysis, plus physical sensors.
selected_metrics = [
    "A5", "A9", "S3", "S7", "S53", "S54", "S172", "S173",
    "S39", "S40", "S41", "S49", "S50", "S125", "S181"
]

metric_summary.filter(
    F.col("metric").isin(selected_metrics)
).orderBy("metric").show(30, truncate=False)

+------+-------+------------------+---------+---------+-----------------------+-----------------------+
|metric|records|days_with_readings|min_value|max_value|first_time_utc         |last_time_utc          |
+------+-------+------------------+---------+---------+-----------------------+-----------------------+
|A5    |217    |148               |0.0      |26688.0  |2020-10-01 00:06:22.759|2021-03-31 16:09:21.608|
|A9    |305    |157               |0.0      |16384.0  |2020-10-01 00:06:22.759|2021-03-31 15:13:51.329|
|S125  |323040 |175               |0.0      |100.0    |2020-10-01 00:00:58.92 |2021-03-31 23:59:57.606|
|S172  |316814 |175               |0.0      |1.0      |2020-10-01 00:01:02.201|2021-03-31 23:59:31.068|
|S173  |316812 |175               |0.0      |1.0      |2020-10-01 00:01:02.201|2021-03-31 23:59:31.068|
|S181  |316815 |175               |0.0      |100.0    |2020-10-01 00:01:02.201|2021-03-31 23:59:31.068|
|S3    |323040 |175               |0.0      |450.0    |2020-10-0

In [16]:
temperature_metrics = ["S3", "S39", "S40", "S41"]

temperature_summary = (
    sw088_data
    .filter(F.col("metric").isin(temperature_metrics))
    .groupBy("metric")
    .agg(
        F.expr(
            "percentile_approx(value, array(0.01, 0.5, 0.99))"
        ).alias("percentiles_01_50_99"),
        F.sum(
            F.when(F.col("value") == 0, 1).otherwise(0)
        ).alias("zero_records"),
        F.sum(
            F.when(
                F.col("value") != F.floor("value"), 1
            ).otherwise(0)
        ).alias("fractional_records"),
        F.count("*").alias("records"),
    )
    .orderBy("metric")
)

temperature_summary.show(truncate=False)

+------+---------------------+------------+------------------+-------+
|metric|percentiles_01_50_99 |zero_records|fractional_records|records|
+------+---------------------+------------+------------------+-------+
|S3    |[100.0, 150.0, 325.0]|8           |0                 |323040 |
|S39   |[86.0, 152.0, 397.0] |11          |0                 |323042 |
|S40   |[86.0, 153.0, 391.0] |11          |0                 |323042 |
|S41   |[13.0, 111.0, 246.0] |79          |0                 |323041 |
+------+---------------------+------------+------------------+-------+



In [17]:
sensor_window = Window.partitionBy("metric").orderBy("when")

sensor_gaps = (
    sw088_data
    .filter(F.col("metric").isin(
        "S3", "S39", "S40", "S41", "S125", "S181", "S172", "S173"
    ))
    .withColumn(
        "gap_seconds",
        (F.col("when") - F.lag("when").over(sensor_window)) / 1000
    )
)

(
    sensor_gaps.groupBy("metric")
    .agg(
        F.expr(
            "percentile_approx(gap_seconds, array(0.5, 0.95, 0.99))"
        ).alias("gap_seconds_p50_p95_p99"),
        F.max("gap_seconds").alias("max_gap_seconds"),
        F.sum(
            F.when(F.col("gap_seconds") > 3600, 1).otherwise(0)
        ).alias("gaps_over_1_hour"),
        F.sum(
            F.when(F.col("gap_seconds") == 0, 1).otherwise(0)
        ).alias("same_timestamp_pairs"),
    )
    .orderBy("metric")
    .show(truncate=False)
)

+------+-------------------------+---------------+----------------+--------------------+
|metric|gap_seconds_p50_p95_p99  |max_gap_seconds|gaps_over_1_hour|same_timestamp_pairs|
+------+-------------------------+---------------+----------------+--------------------+
|S125  |[30.424, 67.262, 169.003]|722055.485     |153             |0                   |
|S172  |[30.48, 67.682, 180.303] |724220.082     |157             |0                   |
|S173  |[30.48, 67.682, 180.303] |724220.082     |157             |0                   |
|S181  |[30.48, 67.682, 180.303] |724220.082     |157             |0                   |
|S3    |[30.424, 67.262, 169.003]|722055.485     |153             |0                   |
|S39   |[30.424, 67.262, 169.003]|722055.485     |153             |0                   |
|S40   |[30.424, 67.262, 169.003]|722055.485     |153             |0                   |
|S41   |[30.424, 67.262, 169.003]|722055.485     |153             |0                   |
+------+-------------

In [18]:
feature_metrics = [
    "S3", "S39", "S40", "S41",
    "S125", "S181", "S172", "S173"
]

hourly_sensor_features = (
    sw088_data
    .filter(F.col("metric").isin(feature_metrics))
    .withColumn("timestamp", F.timestamp_millis("when"))
    .groupBy(
        "hwid",
        "metric",
        F.window("timestamp", "1 hour").alias("time_window")
    )
    .agg(
        F.count("*").alias("n_readings"),
        F.countDistinct(
            F.date_trunc("minute", "timestamp")
        ).alias("minutes_with_readings"),
        F.avg("value").alias("mean_value"),
        F.min("value").alias("min_value"),
        F.max("value").alias("max_value"),
        F.stddev_samp("value").alias("std_value"),
    )
    .select(
        "hwid", "metric",
        F.col("time_window.start").alias("window_start"),
        F.col("time_window.end").alias("window_end"),
        "n_readings", "minutes_with_readings",
        "mean_value", "min_value", "max_value", "std_value"
    )
)

# Summarise coverage before choosing a quality threshold.
(
    hourly_sensor_features.groupBy("metric")
    .agg(
        F.count("*").alias("hours_with_readings"),
        F.expr(
            "percentile_approx(minutes_with_readings, array(0.1, 0.5, 0.9))"
        ).alias("minutes_recorded_p10_p50_p90"),
    )
    .orderBy("metric")
    .show(truncate=False)
)

+------+-------------------+----------------------------+
|metric|hours_with_readings|minutes_recorded_p10_p50_p90|
+------+-------------------+----------------------------+
|S125  |4130               |[25, 53, 58]                |
|S172  |4128               |[24, 52, 57]                |
|S173  |4128               |[24, 52, 57]                |
|S181  |4128               |[24, 52, 57]                |
|S3    |4130               |[25, 53, 58]                |
|S39   |4130               |[25, 53, 58]                |
|S40   |4130               |[25, 53, 58]                |
|S41   |4130               |[25, 53, 58]                |
+------+-------------------+----------------------------+



In [20]:
# Require all eight selected sensors to meet the same threshold.
hour_quality = (
    hourly_sensor_features
    .groupBy("hwid", "window_start", "window_end")
    .agg(
        F.countDistinct("metric").alias("metrics_present"),
        F.min("minutes_with_readings").alias("least_minutes_recorded"),
    )
)

hour_quality.agg(
    F.count("*").alias("hours_with_any_selected_sensor"),
    *[
        F.sum(
            F.when(
                (F.col("metrics_present") == len(feature_metrics))
                & (F.col("least_minutes_recorded") >= threshold),
                1
            ).otherwise(0)
        ).alias(f"hours_all_sensors_at_least_{threshold}_minutes")
        for threshold in (30, 45, 50, 55)
    ],
).show(truncate=False)

+------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+
|hours_with_any_selected_sensor|hours_all_sensors_at_least_30_minutes|hours_all_sensors_at_least_45_minutes|hours_all_sensors_at_least_50_minutes|hours_all_sensors_at_least_55_minutes|
+------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+
|4131                          |3326                                 |2845                                 |2460                                 |1185                                 |
+------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+



In [21]:
alarm_records = (
    sw088_data
    .filter(F.col("metric").isin("A5", "A9"))
    .withColumn("alarm_code", F.col("value").cast("long"))
    .withColumn(
        "overheating",
        F.col("alarm_code").bitwiseAND(32 + 64 + 128) != 0
    )
    .select("hwid", "metric", "when", "alarm_code", "overheating")
    .cache()
)

print("Alarm records cached:", alarm_records.count())

Alarm records cached: 522


In [22]:
qualified_hours = hour_quality.filter(
    (F.col("metrics_present") == len(feature_metrics))
    & (F.col("least_minutes_recorded") >= 45)
)

alarm_examples = (
    alarm_records
    .withColumn("alarm_time", F.timestamp_millis("when"))
    .withColumn(
        "feature_end",
        F.date_trunc("hour", "alarm_time")
    )
    .join(
        qualified_hours,
        (alarm_records.hwid == qualified_hours.hwid)
        & (F.col("feature_end") == qualified_hours.window_end),
        how="inner"
    )
    .select(
        alarm_records.hwid.alias("hwid"),
        "metric", "alarm_time", "overheating",
        "window_start", "window_end"
    )
)

(
    alarm_examples
    .groupBy("metric", "overheating")
    .agg(
        F.count("*").alias("alarm_readings"),
        F.countDistinct("window_end").alias("distinct_feature_hours"),
        F.countDistinct(
            F.to_date("alarm_time")
        ).alias("distinct_alarm_days"),
    )
    .orderBy("metric", "overheating")
    .show(truncate=False)
)

+------+-----------+--------------+----------------------+-------------------+
|metric|overheating|alarm_readings|distinct_feature_hours|distinct_alarm_days|
+------+-----------+--------------+----------------------+-------------------+
|A5    |false      |148           |148                   |126                |
|A5    |true       |19            |10                    |8                  |
|A9    |false      |145           |127                   |101                |
|A9    |true       |93            |52                    |46                 |
+------+-----------+--------------+----------------------+-------------------+



In [23]:
first_alarm_window = (
    Window
    .partitionBy("hwid", "metric", "window_end")
    .orderBy("alarm_time")
)

hour_examples = (
    alarm_examples
    .withColumn(
        "reading_order",
        F.row_number().over(first_alarm_window)
    )
    .filter(F.col("reading_order") == 1)
    .drop("reading_order")
    .withColumn("month", F.date_format("alarm_time", "yyyy-MM"))
)

(
    hour_examples
    .groupBy("month", "metric")
    .agg(
        F.count("*").alias("examples"),
        F.sum(
            F.col("overheating").cast("int")
        ).alias("positive_examples"),
    )
    .orderBy("month", "metric")
    .show(30, truncate=False)
)

+-------+------+--------+-----------------+
|month  |metric|examples|positive_examples|
+-------+------+--------+-----------------+
|2020-10|A5    |28      |5                |
|2020-10|A9    |28      |0                |
|2020-11|A5    |41      |0                |
|2020-11|A9    |44      |0                |
|2020-12|A5    |23      |0                |
|2020-12|A9    |29      |0                |
|2021-01|A5    |25      |1                |
|2021-01|A9    |28      |7                |
|2021-02|A5    |22      |1                |
|2021-02|A9    |28      |27               |
|2021-03|A5    |17      |2                |
|2021-03|A9    |20      |17               |
+-------+------+--------+-----------------+



In [24]:
alarm_order = Window.partitionBy("hwid", "metric").orderBy("when")

observed_onsets = (
    alarm_records
    .withColumn(
        "previous_overheating",
        F.lag("overheating").over(alarm_order)
    )
    .filter(
        (F.col("previous_overheating") == False)
        & (F.col("overheating") == True)
    )
    .withColumn("alarm_time", F.timestamp_millis("when"))
    .withColumn("feature_end", F.date_trunc("hour", "alarm_time"))
    .alias("a")
)

onset_coverage = (
    observed_onsets
    .join(
        hour_quality.alias("q"),
        (F.col("a.hwid") == F.col("q.hwid"))
        & (F.col("a.feature_end") == F.col("q.window_end")),
        "left"
    )
    .select(
        "a.metric",
        "a.alarm_time",
        "q.metrics_present",
        "q.least_minutes_recorded"
    )
    .orderBy("alarm_time", "metric")
)

onset_coverage.show(100, truncate=False)

+------+-----------------------+---------------+----------------------+
|metric|alarm_time             |metrics_present|least_minutes_recorded|
+------+-----------------------+---------------+----------------------+
|A5    |2020-12-16 09:44:32.224|NULL           |NULL                  |
|A9    |2021-01-11 14:11:41.067|8              |51                    |
|A9    |2021-01-12 12:24:50.988|8              |55                    |
|A9    |2021-01-26 09:41:55.998|8              |12                    |
|A5    |2021-01-26 09:44:27.942|8              |12                    |
|A5    |2021-02-02 15:19:05.465|8              |42                    |
|A9    |2021-02-02 17:16:22.941|8              |30                    |
|A5    |2021-02-03 16:11:45.931|8              |34                    |
|A9    |2021-02-03 16:51:43.943|8              |34                    |
|A5    |2021-02-05 14:49:47.226|8              |54                    |
|A5    |2021-02-05 15:40:13.403|8              |56              

In [25]:
onset_coverage.agg(
    F.count("*").alias("observed_circuit_transitions"),
    *[
        F.sum(
            F.when(
                (F.col("metrics_present") == 8)
                & (F.col("least_minutes_recorded") >= threshold),
                1
            ).otherwise(0)
        ).alias(f"transitions_at_{threshold}_minutes")
        for threshold in (30, 45, 50)
    ],
).show(truncate=False)

+----------------------------+-------------------------+-------------------------+-------------------------+
|observed_circuit_transitions|transitions_at_30_minutes|transitions_at_45_minutes|transitions_at_50_minutes|
+----------------------------+-------------------------+-------------------------+-------------------------+
|25                          |17                       |11                       |11                       |
+----------------------------+-------------------------+-------------------------+-------------------------+



In [26]:
examples = hour_examples.alias("e")
history = alarm_records.alias("h")

history_candidates = (
    examples.join(
        history,
        (F.col("e.hwid") == F.col("h.hwid"))
        & (F.col("e.metric") == F.col("h.metric"))
        & (
            F.timestamp_millis(F.col("h.when"))
            < F.col("e.window_end")
        ),
        "left"
    )
    .select(
        "e.*",
        F.col("h.when").alias("previous_alarm_ms"),
        F.col("h.overheating").alias("persistence_prediction"),
    )
)

latest_history = (
    Window.partitionBy("hwid", "metric", "window_end")
    .orderBy(F.col("previous_alarm_ms").desc())
)

baseline_examples = (
    history_candidates
    .withColumn("history_rank", F.row_number().over(latest_history))
    .filter(F.col("history_rank") == 1)
    .drop("history_rank")
    .withColumn(
        "previous_alarm_age_hours",
        (
            F.unix_millis("window_end") - F.col("previous_alarm_ms")
        ) / 3_600_000
    )
)

baseline_examples.groupBy("metric").agg(
    F.count("*").alias("examples"),
    F.count("previous_alarm_ms").alias("examples_with_history"),
    F.expr(
        "percentile_approx(previous_alarm_age_hours, array(0.5, 0.9))"
    ).alias("history_age_hours_p50_p90"),
).show(truncate=False)

+------+--------+---------------------+----------------------------------------+
|metric|examples|examples_with_history|history_age_hours_p50_p90               |
+------+--------+---------------------+----------------------------------------+
|A5    |156     |156                  |[23.75127722222222, 47.56812361111111]  |
|A9    |177     |177                  |[23.647366944444446, 24.478404444444443]|
+------+--------+---------------------+----------------------------------------+



In [27]:
# Initial chronological split:
# October–December: training
# January–March: evaluation
#
# Split using when the features become available.
baseline_test = baseline_examples.filter(
    F.col("window_end") >= F.lit("2021-01-01").cast("timestamp")
)

(
    baseline_test
    .groupBy(
        "metric",
        F.col("overheating").alias("actual"),
        F.col("persistence_prediction").alias("predicted")
    )
    .count()
    .orderBy("metric", "actual", "predicted")
    .show(truncate=False)
)

+------+------+---------+-----+
|metric|actual|predicted|count|
+------+------+---------+-----+
|A5    |false |false    |57   |
|A5    |false |true     |3    |
|A5    |true  |false    |3    |
|A5    |true  |true     |1    |
|A9    |false |false    |21   |
|A9    |false |true     |4    |
|A9    |true  |false    |4    |
|A9    |true  |true     |47   |
+------+------+---------+-----+



In [28]:
split_examples = baseline_examples.withColumn(
    "split",
    F.when(
        F.col("window_end") < F.lit("2021-02-01").cast("timestamp"),
        "train"
    ).when(
        F.col("window_end") < F.lit("2021-03-01").cast("timestamp"),
        "validation"
    ).otherwise("test")
)

split_examples.groupBy("split", "metric").agg(
    F.count("*").alias("examples"),
    F.sum(F.col("overheating").cast("int")).alias("positives"),
    F.sum((~F.col("overheating")).cast("int")).alias("negatives"),
).orderBy("split", "metric").show(truncate=False)

+----------+------+--------+---------+---------+
|split     |metric|examples|positives|negatives|
+----------+------+--------+---------+---------+
|test      |A5    |17      |2        |15       |
|test      |A9    |20      |17       |3        |
|train     |A5    |117     |6        |111      |
|train     |A9    |129     |7        |122      |
|validation|A5    |22      |1        |21       |
|validation|A9    |28      |27       |1        |
+----------+------+--------+---------+---------+



In [29]:
# One row per feature hour, one mean-value column per sensor.
sensor_means = (
    hourly_sensor_features
    .groupBy("hwid", "window_start", "window_end")
    .pivot("metric", feature_metrics)
    .agg(F.first("mean_value"))
)

model_data = (
    split_examples
    .join(
        sensor_means,
        on=["hwid", "window_start", "window_end"],
        how="inner"
    )
    .withColumn("label", F.col("overheating").cast("double"))
    .cache()
)

print("Model examples:", model_data.count())

model_data.select(
    "metric", "split", "window_end", "label", *feature_metrics
).orderBy("window_end", "metric").show(5, truncate=False)

Model examples: 333
+------+-----+-------------------+-----+-----+------------------+------------------+------------------+----+----+----+----+
|metric|split|window_end         |label|S3   |S39               |S40               |S41               |S125|S181|S172|S173|
+------+-----+-------------------+-----+-----+------------------+------------------+------------------+----+----+----+----+
|A5    |train|2020-10-02 00:00:00|1.0  |125.0|101.49090909090908|101.54545454545455|190.65454545454546|0.0 |0.0 |1.0 |0.0 |
|A9    |train|2020-10-02 00:00:00|0.0  |125.0|101.49090909090908|101.54545454545455|190.65454545454546|0.0 |0.0 |1.0 |0.0 |
|A5    |train|2020-10-03 00:00:00|1.0  |125.0|103.41379310344827|103.67241379310344|213.51724137931035|0.0 |0.0 |1.0 |0.0 |
|A9    |train|2020-10-03 00:00:00|0.0  |125.0|103.41379310344827|103.67241379310344|213.51724137931035|0.0 |0.0 |1.0 |0.0 |
|A5    |train|2020-10-05 00:00:00|1.0  |125.0|100.74545454545455|100.94545454545455|166.2             |0.0 |0.0 

In [30]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression

models = {}
prediction_tables = []

for circuit in ("A5", "A9"):
    circuit_data = model_data.filter(F.col("metric") == circuit)
    train = circuit_data.filter(F.col("split") == "train")
    evaluation = circuit_data.filter(F.col("split") != "train")

    pipeline = Pipeline(stages=[
        VectorAssembler(
            inputCols=feature_metrics,
            outputCol="sensor_vector",
            handleInvalid="error"
        ),
        StandardScaler(
            inputCol="sensor_vector",
            outputCol="features",
            withMean=True,
            withStd=True
        ),
        LogisticRegression(
            featuresCol="features",
            labelCol="label",
            regParam=0.1,
            elasticNetParam=0.0,
            maxIter=100,
            threshold=0.5
        )
    ])

    models[circuit] = pipeline.fit(train)
    prediction_tables.append(
        models[circuit].transform(evaluation)
        .select(
            "metric", "split", "window_end",
            "label", "prediction", "probability",
            "persistence_prediction"
        )
    )

model_predictions = prediction_tables[0].unionByName(
    prediction_tables[1]
)

(
    model_predictions
    .groupBy("split", "metric", "label", "prediction")
    .count()
    .orderBy("split", "metric", "label", "prediction")
    .show(30, truncate=False)
)

+----------+------+-----+----------+-----+
|split     |metric|label|prediction|count|
+----------+------+-----+----------+-----+
|test      |A5    |0.0  |0.0       |15   |
|test      |A5    |1.0  |0.0       |2    |
|test      |A9    |0.0  |0.0       |3    |
|test      |A9    |1.0  |0.0       |17   |
|validation|A5    |0.0  |0.0       |21   |
|validation|A5    |1.0  |0.0       |1    |
|validation|A9    |0.0  |0.0       |1    |
|validation|A9    |1.0  |0.0       |24   |
|validation|A9    |1.0  |1.0       |3    |
+----------+------+-----+----------+-----+



In [31]:
comparison = (
    model_predictions
    .withColumn(
        "persistence",
        F.col("persistence_prediction").cast("double")
    )
    .selectExpr(
        "split", "metric", "label",
        """stack(
            2,
            'logistic_regression', prediction,
            'persistence', persistence
        ) as (method, predicted)"""
    )
)

comparison.groupBy("split", "metric", "method").agg(
    F.count("*").alias("examples"),
    F.sum(F.when(
        (F.col("label") == 1) & (F.col("predicted") == 1), 1
    ).otherwise(0)).alias("TP"),
    F.sum(F.when(
        (F.col("label") == 0) & (F.col("predicted") == 1), 1
    ).otherwise(0)).alias("FP"),
    F.sum(F.when(
        (F.col("label") == 1) & (F.col("predicted") == 0), 1
    ).otherwise(0)).alias("FN"),
    F.sum(F.when(
        (F.col("label") == 0) & (F.col("predicted") == 0), 1
    ).otherwise(0)).alias("TN"),
).orderBy("split", "metric", "method").show(truncate=False)

+----------+------+-------------------+--------+---+---+---+---+
|split     |metric|method             |examples|TP |FP |FN |TN |
+----------+------+-------------------+--------+---+---+---+---+
|test      |A5    |logistic_regression|17      |0  |0  |2  |15 |
|test      |A5    |persistence        |17      |0  |2  |2  |13 |
|test      |A9    |logistic_regression|20      |0  |0  |17 |3  |
|test      |A9    |persistence        |20      |16 |3  |1  |0  |
|validation|A5    |logistic_regression|22      |0  |0  |1  |21 |
|validation|A5    |persistence        |22      |0  |1  |1  |20 |
|validation|A9    |logistic_regression|28      |3  |0  |24 |1  |
|validation|A9    |persistence        |28      |26 |1  |1  |0  |
+----------+------+-------------------+--------+---+---+---+---+



In [32]:
class_weights = {}

for circuit in ("A5", "A9"):
    counts = {
        row["label"]: row["count"]
        for row in (
            model_data
            .filter(
                (F.col("metric") == circuit)
                & (F.col("split") == "train")
            )
            .groupBy("label")
            .count()
            .collect()
        )
    }

    total = sum(counts.values())
    class_weights[circuit] = {
        label: total / (2 * count)
        for label, count in counts.items()
    }

    print(circuit, class_weights[circuit])

A5 {1.0: 9.75, 0.0: 0.527027027027027}
A9 {1.0: 9.214285714285714, 0.0: 0.5286885245901639}


In [33]:
weighted_models = {}
weighted_validation_tables = []

for circuit in ("A5", "A9"):
    circuit_data = model_data.filter(F.col("metric") == circuit)

    weights = class_weights[circuit]
    train = (
        circuit_data.filter(F.col("split") == "train")
        .withColumn(
            "class_weight",
            F.when(F.col("label") == 1, F.lit(weights[1.0]))
            .otherwise(F.lit(weights[0.0]))
        )
    )

    validation = circuit_data.filter(F.col("split") == "validation")

    pipeline = Pipeline(stages=[
        VectorAssembler(
            inputCols=feature_metrics,
            outputCol="sensor_vector",
            handleInvalid="error"
        ),
        StandardScaler(
            inputCol="sensor_vector",
            outputCol="features",
            withMean=True,
            withStd=True
        ),
        LogisticRegression(
            featuresCol="features",
            labelCol="label",
            weightCol="class_weight",
            regParam=0.1,
            elasticNetParam=0.0,
            maxIter=100,
            threshold=0.5
        )
    ])

    weighted_models[circuit] = pipeline.fit(train)

    weighted_validation_tables.append(
        weighted_models[circuit].transform(validation)
        .select(
            "metric", "window_end",
            "label", "prediction", "probability"
        )
    )

weighted_validation = weighted_validation_tables[0].unionByName(
    weighted_validation_tables[1]
)

weighted_validation.groupBy(
    "metric", "label", "prediction"
).count().orderBy(
    "metric", "label", "prediction"
).show(truncate=False)

+------+-----+----------+-----+
|metric|label|prediction|count|
+------+-----+----------+-----+
|A5    |0.0  |0.0       |12   |
|A5    |0.0  |1.0       |9    |
|A5    |1.0  |1.0       |1    |
|A9    |0.0  |1.0       |1    |
|A9    |1.0  |0.0       |2    |
|A9    |1.0  |1.0       |25   |
+------+-----+----------+-----+



In [34]:
weighted_test_tables = []

for circuit in ("A5", "A9"):
    test = model_data.filter(
        (F.col("metric") == circuit)
        & (F.col("split") == "test")
    )

    weighted_test_tables.append(
        weighted_models[circuit].transform(test)
        .select("metric", "window_end", "label", "prediction")
    )

weighted_test = weighted_test_tables[0].unionByName(
    weighted_test_tables[1]
)

weighted_test.groupBy(
    "metric", "label", "prediction"
).count().orderBy(
    "metric", "label", "prediction"
).show(truncate=False)

+------+-----+----------+-----+
|metric|label|prediction|count|
+------+-----+----------+-----+
|A5    |0.0  |0.0       |9    |
|A5    |0.0  |1.0       |6    |
|A5    |1.0  |1.0       |2    |
|A9    |0.0  |0.0       |3    |
|A9    |1.0  |0.0       |16   |
|A9    |1.0  |1.0       |1    |
+------+-----+----------+-----+



In [35]:
import csv
from pathlib import Path

results_path = Path(
    "/home/jovyan/work/benchmarks/results/alarm_state_baselines_v1.csv"
)
results_path.parent.mkdir(parents=True, exist_ok=True)

# Counts from the evaluated models.
# TP/FP/FN/TN refer to observed alarm states, not new incident onsets.
rows = [
    ("validation", "A5", "persistence",         0, 1, 1, 20),
    ("validation", "A5", "logistic_regression", 0, 0, 1, 21),
    ("validation", "A5", "weighted_logistic",   1, 9, 0, 12),
    ("validation", "A9", "persistence",         26, 1, 1, 0),
    ("validation", "A9", "logistic_regression", 3, 0, 24, 1),
    ("validation", "A9", "weighted_logistic",   25, 1, 2, 0),
    ("test", "A5", "persistence",               0, 2, 2, 13),
    ("test", "A5", "logistic_regression",       0, 0, 2, 15),
    ("test", "A5", "weighted_logistic",         2, 6, 0, 9),
    ("test", "A9", "persistence",               16, 3, 1, 0),
    ("test", "A9", "logistic_regression",       0, 0, 17, 3),
    ("test", "A9", "weighted_logistic",         1, 0, 16, 3),
]

with results_path.open("x", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["split", "circuit", "method", "TP", "FP", "FN", "TN"])
    writer.writerows(rows)

print("Saved:", results_path)

Saved: /home/jovyan/work/benchmarks/results/alarm_state_baselines_v1.csv


In [36]:
# Build one row per hour with four statistics per sensor.
feature_stats = {
    "mean": "mean_value",
    "min": "min_value",
    "max": "max_value",
    "std": "std_value",
}

expanded_features = (
    hourly_sensor_features
    .groupBy("hwid", "window_start", "window_end")
    .agg(*[
        F.first(
            F.when(F.col("metric") == metric, F.col(source)),
            ignorenulls=True
        ).alias(f"{metric}_{stat}")
        for metric in feature_metrics
        for stat, source in feature_stats.items()
    ])
)

expanded_feature_names = [
    f"{metric}_{stat}"
    for metric in feature_metrics
    for stat in feature_stats
]

expanded_model_data = (
    split_examples
    .join(
        expanded_features,
        on=["hwid", "window_start", "window_end"],
        how="inner"
    )
    .withColumn("label", F.col("overheating").cast("double"))
    .cache()
)

# Check the expanded features before training.
expanded_model_data.agg(
    F.count("*").alias("examples"),
    *[
        F.count(
            F.when(F.col(name).isNull() | F.isnan(name), 1)
        ).alias(f"missing_{name}")
        for name in expanded_feature_names
    ]
).show(truncate=False)

+--------+---------------+--------------+--------------+--------------+----------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+----------------+---------------+---------------+---------------+-----------------+----------------+----------------+----------------+-----------------+----------------+----------------+----------------+-----------------+----------------+----------------+----------------+-----------------+----------------+----------------+----------------+
|examples|missing_S3_mean|missing_S3_min|missing_S3_max|missing_S3_std|missing_S39_mean|missing_S39_min|missing_S39_max|missing_S39_std|missing_S40_mean|missing_S40_min|missing_S40_max|missing_S40_std|missing_S41_mean|missing_S41_min|missing_S41_max|missing_S41_std|missing_S125_mean|missing_S125_min|missing_S125_max|missing_S125_std|missing_S181_mean|missing_S181_min|missing_S181_max|missing_S181_std|missing_S172_mean|missing_S172_min|missing_S172_max|m

In [37]:
usable_features = {}

for circuit in ("A5", "A9"):
    train = expanded_model_data.filter(
        (F.col("metric") == circuit)
        & (F.col("split") == "train")
    )

    deviations = train.agg(*[
        F.stddev_samp(name).alias(name)
        for name in expanded_feature_names
    ]).first().asDict()

    usable_features[circuit] = [
        name
        for name, std in deviations.items()
        if std is not None and std > 0
    ]

    removed = [
        name for name in expanded_feature_names
        if name not in usable_features[circuit]
    ]

    print(f"{circuit}: keeping {len(usable_features[circuit])} features")
    print("Constant features removed:", removed)

A5: keeping 32 features
Constant features removed: []
A9: keeping 32 features
Constant features removed: []


In [38]:
expanded_models = {}
expanded_validation_tables = []

for circuit in ("A5", "A9"):
    circuit_data = expanded_model_data.filter(
        F.col("metric") == circuit
    )
    weights = class_weights[circuit]

    train = (
        circuit_data.filter(F.col("split") == "train")
        .withColumn(
            "class_weight",
            F.when(F.col("label") == 1, F.lit(weights[1.0]))
            .otherwise(F.lit(weights[0.0]))
        )
    )

    pipeline = Pipeline(stages=[
        VectorAssembler(
            inputCols=usable_features[circuit],
            outputCol="sensor_vector",
            handleInvalid="error"
        ),
        StandardScaler(
            inputCol="sensor_vector",
            outputCol="features",
            withMean=True,
            withStd=True
        ),
        LogisticRegression(
            featuresCol="features",
            labelCol="label",
            weightCol="class_weight",
            regParam=0.1,
            elasticNetParam=0.0,
            maxIter=100,
            threshold=0.5
        )
    ])

    expanded_models[circuit] = pipeline.fit(train)

    expanded_validation_tables.append(
        expanded_models[circuit]
        .transform(circuit_data.filter(F.col("split") == "validation"))
        .select("metric", "window_end", "label", "prediction")
    )

expanded_validation = expanded_validation_tables[0].unionByName(
    expanded_validation_tables[1]
)

expanded_validation.groupBy(
    "metric", "label", "prediction"
).count().orderBy(
    "metric", "label", "prediction"
).show(truncate=False)

+------+-----+----------+-----+
|metric|label|prediction|count|
+------+-----+----------+-----+
|A5    |0.0  |0.0       |16   |
|A5    |0.0  |1.0       |5    |
|A5    |1.0  |1.0       |1    |
|A9    |0.0  |1.0       |1    |
|A9    |1.0  |0.0       |6    |
|A9    |1.0  |1.0       |21   |
+------+-----+----------+-----+



In [39]:
expanded_test_tables = []

for circuit in ("A5", "A9"):
    test = expanded_model_data.filter(
        (F.col("metric") == circuit)
        & (F.col("split") == "test")
    )

    expanded_test_tables.append(
        expanded_models[circuit].transform(test)
        .select("metric", "window_end", "label", "prediction")
    )

expanded_test = expanded_test_tables[0].unionByName(
    expanded_test_tables[1]


)

expanded_test.groupBy(
    "metric", "label", "prediction"
).count().orderBy(
    "metric", "label", "prediction"
).show(truncate=False)


+------+-----+----------+-----+
|metric|label|prediction|count|
+------+-----+----------+-----+
|A5    |0.0  |0.0       |11   |
|A5    |0.0  |1.0       |4    |
|A5    |1.0  |1.0       |2    |
|A9    |0.0  |0.0       |3    |
|A9    |1.0  |0.0       |16   |
|A9    |1.0  |1.0       |1    |
+------+-----+----------+-----+



In [40]:
a9_sensor_comparison = (
    model_data
    .filter(F.col("metric") == "A9")
    .groupBy("split", "label")
    .agg(
        F.count("*").alias("examples"),
        *[
            F.round(F.avg(name), 2).alias(f"avg_{name}")
            for name in feature_metrics
        ]
    )
    .orderBy("split", "label")
)

a9_sensor_comparison.show(truncate=False)

+----------+-----+--------+------+-------+-------+-------+--------+--------+--------+--------+
|split     |label|examples|avg_S3|avg_S39|avg_S40|avg_S41|avg_S125|avg_S181|avg_S172|avg_S173|
+----------+-----+--------+------+-------+-------+-------+--------+--------+--------+--------+
|test      |0.0  |3       |125.89|142.81 |137.06 |257.84 |4.24    |18.0    |0.99    |0.01    |
|test      |1.0  |17      |154.58|185.32 |181.43 |118.4  |0.0     |0.3     |1.0     |0.0     |
|train     |0.0  |122     |132.66|137.07 |136.98 |125.35 |5.92    |4.42    |0.4     |0.6     |
|train     |1.0  |7       |257.31|250.53 |251.11 |73.87  |30.91   |14.13   |0.57    |0.43    |
|validation|0.0  |1       |325.0 |387.0  |387.94 |143.09 |22.21   |0.0     |1.0     |0.0     |
|validation|1.0  |27      |303.88|326.73 |327.05 |109.88 |12.06   |0.07    |1.0     |0.0     |
+----------+-----+--------+------+-------+-------+-------+--------+--------+--------+--------+



In [41]:
from pyspark.ml.functions import vector_to_array

a9_diagnostic = (
    expanded_models["A9"]
    .transform(
        expanded_model_data.filter(F.col("metric") == "A9")
    )
    .withColumn(
        "positive_score",
        vector_to_array("probability")[1]
    )
    .select(
        "split", "alarm_time", "window_end",
        "label", "prediction",
        F.round("positive_score", 3).alias("positive_score"),
        "persistence_prediction",
        "previous_alarm_age_hours",
        "S3_mean", "S125_mean", "S181_mean"
    )
)

a9_diagnostic.filter(
    F.col("label") == 1
).orderBy("alarm_time").show(100, truncate=False)


+----------+-----------------------+-------------------+-----+----------+--------------+----------------------+------------------------+------------------+-----------------+------------------+
|split     |alarm_time             |window_end         |label|prediction|positive_score|persistence_prediction|previous_alarm_age_hours|S3_mean           |S125_mean        |S181_mean         |
+----------+-----------------------+-------------------+-----+----------+--------------+----------------------+------------------------+------------------+-----------------+------------------+
|train     |2021-01-11 14:11:41.067|2021-01-11 14:00:00|1.0  |1.0       |0.886         |false                 |37.92634972222222       |150.0             |88.7578947368421 |97.04347826086956 |
|train     |2021-01-12 12:24:50.988|2021-01-12 12:00:00|1.0  |1.0       |0.873         |false                 |3.671943333333333       |176.09174311926606|66.0             |1.8333333333333333|
|train     |2021-01-13 00:10:15.184

In [42]:
alarm_order = Window.partitionBy("hwid", "metric").orderBy("when")
alarm_history = alarm_order.rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

alarm_spells = (
    alarm_records
    .withColumn(
        "previous_state",
        F.lag("overheating").over(alarm_order)
    )
    .withColumn(
        "starts_active_spell",
        F.when(
            F.col("overheating")
            & (
                F.col("previous_state").isNull()
                | (F.col("previous_state") == False)
            ),
            1
        ).otherwise(0)
    )
    .withColumn(
        "observed_spell_id",
        F.sum("starts_active_spell").over(alarm_history)
    )
)

(
    alarm_spells
    .filter(F.col("overheating"))
    .groupBy("metric", "observed_spell_id")
    .agg(
        F.timestamp_millis(F.min("when")).alias("first_active_reading"),
        F.timestamp_millis(F.max("when")).alias("last_active_reading"),
        F.count("*").alias("active_readings"),
    )
    .orderBy("metric", "observed_spell_id")
    .show(50, truncate=False)
)

+------+-----------------+-----------------------+-----------------------+---------------+
|metric|observed_spell_id|first_active_reading   |last_active_reading    |active_readings|
+------+-----------------+-----------------------+-----------------------+---------------+
|A5    |1                |2020-10-01 00:06:22.759|2020-10-06 07:10:38.626|14             |
|A5    |2                |2020-12-16 09:44:32.224|2020-12-16 09:44:32.224|1              |
|A5    |3                |2021-01-26 09:44:27.942|2021-01-26 10:42:30.341|3              |
|A5    |4                |2021-02-02 15:19:05.465|2021-02-02 15:19:05.465|1              |
|A5    |5                |2021-02-03 16:11:45.931|2021-02-03 16:52:12.823|3              |
|A5    |6                |2021-02-05 14:49:47.226|2021-02-05 14:49:47.226|1              |
|A5    |7                |2021-02-05 15:40:13.403|2021-02-05 15:56:56.426|5              |
|A5    |8                |2021-03-29 09:41:15.35 |2021-03-29 10:10:55.567|3              |

In [43]:
spell_lookup = alarm_spells.select(
    "hwid", "metric",
    F.timestamp_millis("when").alias("alarm_time"),
    F.when(
        F.col("overheating"),
        F.col("observed_spell_id")
    ).alias("active_spell_id")
)

examples_with_spells = split_examples.join(
    spell_lookup,
    on=["hwid", "metric", "alarm_time"],
    how="left"
)

examples_with_spells.groupBy("split", "metric").agg(
    F.count("*").alias("examples"),
    F.sum(F.col("overheating").cast("int")).alias("positive_examples"),
    F.countDistinct("active_spell_id").alias("observed_active_spells"),
).orderBy("split", "metric").show(truncate=False)

+----------+------+--------+-----------------+----------------------+
|split     |metric|examples|positive_examples|observed_active_spells|
+----------+------+--------+-----------------+----------------------+
|test      |A5    |17      |2                |2                     |
|test      |A9    |20      |17               |3                     |
|train     |A5    |117     |6                |2                     |
|train     |A9    |129     |7                |3                     |
|validation|A5    |22      |1                |1                     |
|validation|A9    |28      |27               |5                     |
+----------+------+--------+-----------------+----------------------+



In [44]:
# Identify active spells represented in more than one split.
crossing_spells = (
    examples_with_spells
    .filter(F.col("active_spell_id").isNotNull())
    .groupBy("hwid", "metric", "active_spell_id")
    .agg(F.countDistinct("split").alias("split_count"))
    .filter(F.col("split_count") > 1)
    .select("hwid", "metric", "active_spell_id")
)

# Remove examples belonging to those spells.
# Negative examples have null spell IDs and remain in the table.
spell_separated_examples = examples_with_spells.join(
    crossing_spells,
    on=["hwid", "metric", "active_spell_id"],
    how="left_anti"
)

spell_separated_examples.groupBy("split", "metric").agg(
    F.count("*").alias("examples"),
    F.sum(F.col("overheating").cast("int")).alias("positives"),
    F.countDistinct("active_spell_id").alias("active_spells"),
).orderBy("split", "metric").show(truncate=False)


+----------+------+--------+---------+-------------+
|split     |metric|examples|positives|active_spells|
+----------+------+--------+---------+-------------+
|test      |A5    |17      |2        |2            |
|test      |A9    |5       |2        |2            |
|train     |A5    |117     |6        |2            |
|train     |A9    |125     |3        |2            |
|validation|A5    |22      |1        |1            |
|validation|A9    |5       |4        |3            |
+----------+------+--------+---------+-------------+

